# Zava Voice-of-Customer: Semantic Theme Discovery & Retrieval Audit

**Mission:** Principal Data Scientist

**Goal:** Discover customer themes across multilingual Reviews and SupportChats using precomputed vector embeddings, then audit how trustworthy the retrieved matches actually are (precision@k + false-positive analysis).

**Agent / model:** GitHub Copilot CLI in agent mode — model selection: `Auto` (Copilot auto-routed the model; exact backing model not pinned, noted for reproducibility).

**Tools used:** SQL MCP (`describe_entities`, `read_records`, `aggregate_records`) + custom vector tool `FindSimilarDocsByDocId`.

## 1. Corpus exploration

Counts from `describe_entities` + `read_records` (Turn 2).

| Metric | Value |
| --- | --- |
| Total documents (Docs) | 89 |
| Customer Reviews | 44 |
| Support Chats | 45 |
| SupportChats transcripts | 45 |
| SupportTickets | 47 |
| Customers | 1,228 |
| SalesOrders | 1,800 |
| SalesOrderLines | 2,747 |
| Languages (support transcripts) | EN 28 / ES 13 / FR 4 |

Docs is near-evenly split between reviews (44) and support chats (45), so a theme can cross both feedback and conversational support. Language mix is English-dominant and unbalanced — important context when judging cross-language retrieval. At 89 docs the corpus is sufficient for initial theme discovery, not statistical generalization.

## 2. Seed document selection

Two representative seeds chosen across sentiment and language:

- **English seed — DocId 2:** negative review, *"Disappointed with the quality of this top"*
- **Spanish seed — DocId 9:** positive review, *"Excelente ajuste y muy cómodo, aunque llegó con un pequeño defecto"*

## 3. Vector similarity retrieval

`EXEC dbo.FindSimilarDocsByDocId @DocId = 2, @TopN = 5;`

**English seed (DocId 2) neighbors:**

| Rank | DocId | Title | Sentiment | Cosine dist |
| --- | --- | --- | --- | --- |
| 1 | 7 | Disappointed and want a refund | negative | 0.2377 |
| 2 | 18 | Great look and fit, even with a small defect | **positive** | 0.3197 |
| 3 | 21 | Disappointed with the fit and finish | negative | 0.3268 |
| 4 | 3 | Disappointed and asking for a refund | negative | 0.3392 |

**Spanish seed (DocId 9) neighbors:**

| Rank | DocId | Title | Sentiment | Cosine dist |
| --- | --- | --- | --- | --- |
| 1 | 15 | Buena compra, cómoda y con buen ajuste | positive | 0.2024 |
| 2 | 25 | Muy bonita, aunque llegó con un pequeño defecto | positive | 0.2050 |
| 3 | 18 | Great look and fit, even with a small defect | positive (EN) | 0.2751 |
| 4 | 37 | Muy buena compra, cómoda y con buen ajuste | positive | 0.3271 |

**Cross-language bridge:** DocId 18 (English) appears in *both* neighborhoods. It sits closer to the Spanish-positive seed (0.2751) than to the English-negative seed (0.3197) — quantitative confirmation that it is a stronger thematic match to the positive cluster. The embedding tracks topic + sentiment together for DocId 9, but for DocId 2 the match is topical only.

## 4. Hand-labeled retrieval audit

Each neighbor labeled RELEVANT (shares both topic AND the seed's sentiment) or NOT RELEVANT.

**English NEGATIVE seed (DocId 2) — "Disappointed with the quality of this top":**

| Rank | DocId | Label | Reason (from body text) |
| --- | --- | --- | --- |
| 1 | 7 | RELEVANT | Negative review of a premium top — poor fit, cheap fabric, worn stitching, refund demand. |
| 2 | 18 | NOT RELEVANT | Positive review praising fit and comfort despite only a small quality issue — sentiment mismatch. |
| 3 | 21 | RELEVANT | Negative review — fit, comfort, quality complaints on workout shorts; didn't live up to the price. |
| 4 | 3 | RELEVANT | Negative review of premium shorts — bad fit, cheap fabric, inconsistent sizing, full-refund request. |

**precision@4 = 0.75** (3/4). Miss = DocId 18, analyzed in Section 6.

**Spanish POSITIVE seed (DocId 9) — "Excelente ajuste y muy cómodo...":**

| Rank | DocId | Label | Reason (from body text) |
| --- | --- | --- | --- |
| 1 | 15 | RELEVANT | Spanish positive review praising comfortable fit and good sizing. |
| 2 | 25 | RELEVANT | Spanish positive review — comfortable, fits well, satisfactory despite a small defect. |
| 3 | 18 | RELEVANT | English positive review — fits well, feels comfortable, looks great overall (cross-language match). |
| 4 | 37 | RELEVANT | Spanish positive review — size fits well, good fabric quality, comfortable all day. |

**precision@4 = 1.00** (4/4).

**Key cross-seed finding:** the positive cluster retrieves cleanly (1.00) while the negative cluster carries a false positive (0.75). The same document (DocId 18) is a *correct* match for the positive seed but a *wrong* match for the negative seed — the model separates topic well but sentiment poorly, and that weakness shows up specifically when the seed is negative.

In [ ]:
# 5. Precision@k
def precision_at_k(labels):
    """labels: list of 'RELEVANT'/'NOT RELEVANT' for ranks 1..k"""
    k = len(labels)
    hits = sum(1 for l in labels if l == 'RELEVANT')
    return hits / k

# English NEGATIVE seed (DocId 2): 7=R, 18=NOT, 21=R, 3=R
eng_labels = ['RELEVANT', 'NOT RELEVANT', 'RELEVANT', 'RELEVANT']
print(f"precision@4 (DocId 2, negative): {precision_at_k(eng_labels):.2f}")
print(f"precision@3 (DocId 2, negative): {precision_at_k(eng_labels[:3]):.2f}")

# Spanish POSITIVE seed (DocId 9): 15=R, 25=R, 18=R, 37=R
spa_labels = ['RELEVANT', 'RELEVANT', 'RELEVANT', 'RELEVANT']
print(f"precision@4 (DocId 9, positive): {precision_at_k(spa_labels):.2f}")

# Cross-seed: positive retrieves cleanly (1.00), negative carries a false positive (0.75)

## 6. False-positive analysis

**DocId 18 in the English negative neighborhood — characterized false positive.**

- **Shared vocabulary:** product/quality/fit/defect terms ("small defect", "fit", "top")
- **Divergent meaning:** seed wants a refund and is disappointed; DocId 18 is "delighted" and "fantastic buy overall"
- **Source type:** both Reviews
- **Would keyword search make the same mistake?** Yes — keyword overlap is high; the embedding inherits the same topical-but-not-sentiment confusion

**Takeaway:** the embedding captures *topic* strongly but *sentiment polarity* weakly. Same document is a correct neighbor for a positive seed (DocId 9) and a false positive for a negative seed (DocId 2).

## 7. Theme summary

| Theme | Grounding DocIds | Crosses language? | Crosses source? | Confidence |
| --- | --- | --- | --- | --- |
| Product quality disappointment / refund | 2, 3, 7, 21 | [check] | [check] | HIGH |
| Fit & comfort satisfaction | 9, 15, 18, 37 | YES (EN+ES) | [check] | HIGH |
| Minor defect — tolerated vs dealbreaker | 18, 25 + seeds | YES | [check] | MEDIUM |

## 8. Model card

*See `model-card.md` — paste final version here for the submission.*

## 9. Architecture / workflow

```mermaid
flowchart LR
    Docs[(Docs: Reviews + SupportChats\nVECTOR 1536, EN/ES/FR)] --> Seeds[Seed selection\nDocId 2 EN-neg, DocId 9 ES-pos]
    Seeds --> Vec[FindSimilarDocsByDocId\ncosine distance]
    Vec --> Neigh[Top-k neighbors]
    Neigh --> Label[Hand-labeling\nRELEVANT / NOT RELEVANT]
    Label --> Prec[precision@k]
    Label --> FP[False-positive analysis\nDocId 18 sentiment mismatch]
    Prec --> Card[Theme summary + Model card]
    FP --> Card
    Card --> Human[Human review before action]
```

## 10. What I would test or improve next

- Add a sentiment dimension so retrieval can separate polarity (the key failure found)
- Build a labeled gold set for rigorous precision@k
- Balance languages and expand the corpus before trusting cluster sizes
- Keep refund/escalation decisions human-in-the-loop